# Step 9 — Pairwise DEG analysis across all brain regions (Capillary EC, edgeR)

Exhaustive pairwise differential-expression screen across all 33 brain regions in this capillary
EC dataset (528 region pairs = C(33,2)), controlling for donor and AV zonation position
(`AV_bin`), directly extending the pseudobulk-edgeR framework already built in
`06.1.R_DEG_analysis_Capi_rg_clusters_edgeR.ipynb` — same pseudobulk construction
(`Seurat::AggregateExpression`), same `AV_bin` definition (global pseudotime quartiles), same
edgeR pipeline (`DGEList` → `filterByExpr` → `normLibSizes` → `estimateDisp` → `glmQLFit`) and the
same contrast-vector approach, generalized from `06.1`'s "each region_cluster vs the mean of the
rest" (6 one-vs-rest tests) to **every region vs every other region individually** (528 direct
pairwise tests) — and from `region_cluster` (4 coarse groups) to the 33 fine-grained `region_name`
levels, per the request.

**Design:** `expression ~ 0 + region_name + individualID + AV_bin` (region and AV_bin as fixed
effects, matching `06.1`'s own AV-adjusted section; donor as a fixed effect rather than
`(1|individualID)`, also matching `06.1`, not the `dream`/mixed-model convention used in
`02`/`07`/`08`).

**One deliberate deviation from `06.1`:** `06.1`'s own code labels its significance column `FDR`
but actually computes it via `p.adjust(method="bonferroni")`. This notebook uses genuine BH-FDR
per pair instead — matching the rest of this repo (see `CLAUDE.md`) and `edgeR::topTags()`'s own
default — since Bonferroni across ~2,000 genes would be needlessly conservative for a single
pairwise test.

**Deliverables:** (1) heatmaps of the number of DEGs — total, and separately up/down — across all
528 region pairs, grouped both by anatomical category and by the data-driven `region_cluster`;
(2) a functional summary of what those DEGs represent, built two ways: a per-region "recurrent
signature" (genes consistently up in a region across many of its pairwise comparisons) enriched
via GO Biological Process, and a detailed case-study enrichment for the handful of most
transcriptionally distinct region pairs; (3) an overlap of the significant pairwise DEGs against
the curated BBB transport-gene panel to find recurrent, region-specific transport markers; and
(4) a sensitivity check re-running the entire pairwise screen without AV_bin adjustment, to see
whether that covariate is driving the results.

Route: native R (`ir` kernel, `R_4.5`).

## 9.0 Setup, paths, parameters

In [ ]:
options(stringsAsFactors = FALSE)
set.seed(1)
suppressPackageStartupMessages({
  library(Seurat); library(schard); library(stringr); library(dplyr); library(tidyr)
  library(edgeR); library(ggplot2); library(ComplexHeatmap); library(circlize)
  library(clusterProfiler); library(org.Hs.eg.db); library(viridis)
})

DATA <- "../Data"
RES  <- "../Results"
FIG  <- "../Figures"
dir.create(RES, showWarnings = FALSE)
dir.create(FIG, showWarnings = FALSE)

# ---- parameters (safe to tweak and re-run) ----
N_AV_BINS <- 4                # global pseudotime quartiles, matches 06.1
MIN_CELLS_PER_SAMPLE <- 10     # min nuclei for a (donor x region x AV_bin) pseudobulk sample to be built
LIBSIZE_QTILE_CUTOFF <- 0.05   # 06.1's own additional QC: drop samples below this lib.size quantile
LOGFC_THRESH <- 0.5           # matches 06.1's significance convention
FDR_THRESH   <- 0.05           # BH-FDR per pair (see note above re: 06.1's bonferroni)
WIN_FREQ_THRESH <- 8           # a gene counts toward a region's "signature" if up in >=8/32 of its pairwise tests
MIN_SIG_GENES_FOR_GO <- 5      # regions/pairs need at least this many signature genes to run enrichGO
N_CASE_STUDY_PAIRS <- 5        # most-distinct pairs to run detailed pairwise enrichment on
NCORES <- 10

sessionInfo()$R.version$version.string

## 9.1 Load data, AV_bin

Same convention as `06.1`: `AV_bin` is built from **global** pseudotime quartiles (pooled across
all capillary nuclei, not per-region), so it captures each cell's AV zonation position on a common
scale.

In [ ]:
obj_oi <- schard::h5ad2seurat(file.path(DATA, "Capi_temp_processed.h5ad"))
cat("cells x genes:", ncol(obj_oi), "x", nrow(obj_oi), "\n")
cat("n region_name levels:", length(unique(obj_oi$region_name)), "\n")
cat("n donors:", length(unique(obj_oi$individualID)), "\n")

qs <- quantile(obj_oi$pseudotime, probs = seq(0, 1, length.out = N_AV_BINS + 1))
labs <- paste0("AV", seq_len(N_AV_BINS))
obj_oi$AV_bin <- cut(obj_oi$pseudotime, breaks = qs, include.lowest = TRUE, labels = labs)
cat("\nAV_bin sizes:\n"); print(table(obj_oi$AV_bin))

## 9.2 Pseudobulk sample coverage (donor x region x AV_bin)

Splitting each donor x region pseudobulk sample further by `AV_bin` (as `06.1` does) creates up to
33 regions x 4 donors x 4 AV bins = 528 possible pseudobulk cells, many of which will have very
few nuclei. A minimum-cell floor is applied **before** aggregation (a real coverage filter, unlike
`06.1`'s purely relative 5%-quantile library-size filter, which alone would still admit many
near-empty samples given how sparse the finest-grained splits are).

In [ ]:
tab3 <- table(obj_oi$sampleID, obj_oi$AV_bin)
v <- as.vector(tab3)
cat("(donor x region) x AV_bin combos: ", length(v), " (", sum(v>0), " nonzero)\n", sep="")
cat("n zero:", sum(v==0), " n 1-9 cells:", sum(v>0 & v<MIN_CELLS_PER_SAMPLE),
    " n >=", MIN_CELLS_PER_SAMPLE, " cells:", sum(v>=MIN_CELLS_PER_SAMPLE), "\n")

ok_combo <- as.data.frame(tab3) %>% filter(Freq >= MIN_CELLS_PER_SAMPLE)
colnames(ok_combo) <- c("sampleID","AV_bin","n")
obj_oi$combo_id <- paste(obj_oi$sampleID, obj_oi$AV_bin, sep="__")
ok_combo$combo_id <- paste(ok_combo$sampleID, ok_combo$AV_bin, sep="__")
scope <- subset(obj_oi, subset = combo_id %in% ok_combo$combo_id)
cat("\ncells kept:", ncol(scope), "of", ncol(obj_oi),
    "; pseudobulk samples that will be built:", nrow(ok_combo), "\n")

## 9.3 Pseudobulk construction and design matrix

`Seurat::AggregateExpression()` grouped by `(sampleID, region_name, AV_bin)`, matching `06.1`'s
own `AggregateExpression(..., group.by=c("sampleID","region_cluster","AV_bin"))` call exactly,
just with `region_name` (33 levels) in place of `region_cluster` (4-5 levels). Donor is looked up
directly from the original object's `sampleID → individualID` map (more robust than `06.1`'s
string-splitting of the aggregated sample names, which relies on a fixed number of `-`/`_`
delimiters).

In [ ]:
pseudo_obj <- AggregateExpression(scope, assays = "RNA", return.seurat = TRUE,
                                   group.by = c("sampleID","region_name","AV_bin"))
mat  <- pseudo_obj[['RNA']]$counts
meta <- pseudo_obj@meta.data
cat("n pseudobulk samples:", ncol(mat), "\n")

donor_lut <- scope@meta.data %>% distinct(sampleID, individualID)
donor_lut$sampleID_dash <- gsub("_", "-", donor_lut$sampleID)   # AggregateExpression sanitizes "_" to "-"
meta$donor <- donor_lut$individualID[match(meta$sampleID, donor_lut$sampleID_dash)]
stopifnot(sum(is.na(meta$donor)) == 0)

y <- DGEList(counts = mat, group = meta$region_name)
y$samples$donor       <- meta$donor
y$samples$AV_bin       <- meta$AV_bin
y$samples$region_name  <- meta$region_name

## 06.1's own additional QC: drop samples below the 5% lib.size quantile
cutoff <- quantile(y$samples$lib.size, LIBSIZE_QTILE_CUTOFF)
keep.samples <- y$samples$lib.size > cutoff
cat("samples kept by lib.size filter:", sum(keep.samples), "of", length(keep.samples), "\n")
y <- y[, keep.samples]

group  <- droplevels(factor(y$samples$region_name))
donor  <- droplevels(factor(y$samples$donor))
AV_bin <- droplevels(factor(y$samples$AV_bin))
cat("n region levels:", nlevels(group), " | n donors:", nlevels(donor), " | n AV bins:", nlevels(AV_bin), "\n")
cat("samples per region (min/median/max):", min(table(group)), "/", median(table(group)), "/", max(table(group)), "\n")

# region_name -> pretty display name (make.names() mangles spaces to "." in design colnames)
region_display <- levels(group)
names(region_display) <- make.names(region_display)

design <- model.matrix(~0 + group + donor + AV_bin)
colnames(design) <- make.names(colnames(design))
cat("\ndesign: ", nrow(design), "samples x", ncol(design), "columns; rank =", qr(design)$rank,
    ifelse(qr(design)$rank == ncol(design), " (full rank)", " (RANK DEFICIENT)"), "\n")

write.csv(data.frame(region=names(table(group)), n_samples=as.integer(table(group))),
          file.path(RES, "09_pseudobulk_samples_per_region.csv"), row.names = FALSE)

## 9.4 Gene filter, normalization, dispersion, model fit

In [ ]:
keep <- filterByExpr(y, design = design)
cat("genes kept:", sum(keep), "of", length(keep), "\n")
y <- y[keep, , keep.lib.sizes = FALSE]
y <- normLibSizes(y)

y <- estimateDisp(y, design, robust = TRUE)
cat("common dispersion:", y$common.dispersion, "\n")
fit <- glmQLFit(y, design, robust = TRUE)

saveRDS(list(y = y, fit = fit, design = design, region_display = region_display),
        file.path(RES, "09_fit.rds"))

## 9.5 All 528 pairwise contrasts

For every region pair `(A, B)`, a contrast vector (`+1` on `A`'s column, `-1` on `B`'s, `0`
elsewhere, including the donor/AV_bin columns) isolates `A` vs `B` directly — the same
contrast-vector mechanics `06.1` uses for its one-vs-rest tests, just with a two-region contrast
instead of a target-vs-mean-of-rest one. ~11 minutes for all 528 `glmQLFTest()` calls.

In [ ]:
grp_cols <- grep("^group", colnames(design), value = TRUE)
regions  <- sub("^group", "", grp_cols)
K <- length(regions)
pairs <- combn(regions, 2, simplify = FALSE)
cat("K regions:", K, " | n pairs:", length(pairs), "\n")

mk_contrast <- function(r1, r2, design) {
  v <- setNames(rep(0, ncol(design)), colnames(design))
  v[paste0("group", r1)] <- 1
  v[paste0("group", r2)] <- -1
  v
}

t0 <- Sys.time()
summary_rows <- vector("list", length(pairs))
sig_list     <- vector("list", length(pairs))

for (k in seq_along(pairs)) {
  r1 <- pairs[[k]][1]; r2 <- pairs[[k]][2]
  qlf <- glmQLFTest(fit, contrast = mk_contrast(r1, r2, design))
  tt <- topTags(qlf, n = Inf, adjust.method = "BH")$table
  tt$gene <- rownames(tt)
  n_up   <- sum(tt$FDR < FDR_THRESH & tt$logFC >  LOGFC_THRESH)
  n_down <- sum(tt$FDR < FDR_THRESH & tt$logFC < -LOGFC_THRESH)
  summary_rows[[k]] <- data.frame(region1 = region_display[r1], region2 = region_display[r2],
                                   n_up_1v2 = n_up, n_down_1v2 = n_down, n_total = n_up + n_down)
  sig <- tt[tt$FDR < FDR_THRESH & abs(tt$logFC) > LOGFC_THRESH, ]
  if (nrow(sig) > 0) {
    sig$region1 <- region_display[r1]; sig$region2 <- region_display[r2]
    sig$direction <- ifelse(sig$logFC > 0, paste0("up_in_", region_display[r1]), paste0("up_in_", region_display[r2]))
    sig_list[[k]] <- sig
  }
}
cat("total time:", round(as.numeric(Sys.time() - t0, units = "mins"), 1), "min\n")

summary_df <- do.call(rbind, summary_rows)
sig_df     <- do.call(rbind, sig_list)
cat("pairs with 0 DEGs:", sum(summary_df$n_total == 0), "of", nrow(summary_df), "\n")
cat("total significant gene-pair rows:", nrow(sig_df), "\n")
print(summary(summary_df$n_total))

saveRDS(summary_df, file.path(RES, "09_pairwise_summary.rds"))
saveRDS(sig_df,     file.path(RES, "09_pairwise_sig_genes.rds"))
write.csv(summary_df, file.path(RES, "09_pairwise_DEG_summary.csv"), row.names = FALSE)
write.csv(sig_df,     file.path(RES, "09_pairwise_sig_genes.csv"), row.names = FALSE)

## Applying the final significance threshold (|log2FC| > 0.5)

The 528-pair loop above (§9.5) classified genes as significant using `FDR < 0.05` and
`|log2FC| > 0.25`. Tightening the effect-size floor to **0.5** here — the threshold used for every
result from this point on — doesn't require re-running the pairwise fits: the cached `sig_df`
(FDR<0.05 & |log2FC|>0.25) is already a superset of what's needed at |log2FC|>0.5, so refiltering
it and rebuilding `summary_df` from the tightened set is mathematically identical to having used
`LOGFC_THRESH = 0.5` in the loop from the start. `sig_df`/`summary_df` are reassigned in place, so
every section below (heatmaps, GO enrichment, transport-marker overlap, the no-AV sensitivity
check) automatically reflects the tighter threshold.

In [ ]:
## Refiltering to the final significance threshold: |log2FC| > 0.5 (tightened from the 0.25 used
## when the 528-pair loop above ran). This doesn't require redoing the pairwise model fits --
## FDR<0.05 & |logFC|>0.25 (what's cached in sig_df) is already a superset of FDR<0.05 & |logFC|>0.5,
## so refiltering the cached results and rebuilding summary_df from that is exactly equivalent to
## having used LOGFC_THRESH=0.5 in the loop from the start.
LOGFC_THRESH <- 0.5

sig_df <- sig_df %>% filter(abs(logFC) > LOGFC_THRESH)
cat("sig_df rows after tightening to |log2FC| >", LOGFC_THRESH, ":", nrow(sig_df), "\n")

summary_df <- sig_df %>%
  group_by(region1, region2) %>%
  summarise(n_up_1v2   = sum(direction == paste0("up_in_", region1)),
            n_down_1v2 = sum(direction == paste0("up_in_", region2)),
            n_total    = n(), .groups = "drop") %>%
  as.data.frame()
# pairs with zero surviving DEGs at the stricter threshold need to be added back explicitly
all_pairs <- combn(regions, 2, simplify = FALSE)
all_pairs_df <- do.call(rbind, lapply(all_pairs, function(p) data.frame(region1 = region_display[p[1]], region2 = region_display[p[2]])))
summary_df <- merge(all_pairs_df, summary_df, by = c("region1","region2"), all.x = TRUE)
summary_df[is.na(summary_df)] <- 0

cat("pairs with 0 DEGs at the new threshold:", sum(summary_df$n_total == 0), "of", nrow(summary_df), "\n")
cat("total significant gene-pair rows:", sum(summary_df$n_total), "\n")

saveRDS(summary_df, file.path(RES, "09_pairwise_summary.rds"))
saveRDS(sig_df, file.path(RES, "09_pairwise_sig_genes.rds"))
write.csv(summary_df, file.path(RES, "09_pairwise_DEG_summary.csv"), row.names = FALSE)
write.csv(sig_df, file.path(RES, "09_pairwise_sig_genes.csv"), row.names = FALSE)


## 9.6 Heatmaps — number of DEGs across all region pairs

Three 33x33 matrices: **total** DEGs per pair (symmetric — the overall "how transcriptionally
distinct are these two regions" summary), and the **up**/**down** breakdown as a directional
matrix (`M[row, col]` = genes significantly higher in the row region than the column region;
the down-regulated matrix is its transpose, shown explicitly for direct readability). Regions are
grouped by `Region_layer_2` (from `Data/region_update.csv`) so anatomically related regions sit
together and block structure is visible.

In [ ]:
region_meta <- read.csv(file.path(DATA, "region_update.csv"))
regions_pretty <- region_display[regions]
layer2 <- region_meta$Region_layer_2[match(regions_pretty, region_meta$Region_layer_1)]
names(layer2) <- regions_pretty
ord <- order(layer2, regions_pretty)
regions_ord <- regions_pretty[ord]
layer2_ord  <- layer2[ord]
cat("Region_layer_2 categories:\n"); print(table(layer2_ord))

# region_cluster, kept in the SAME region_layer-based order (not reordered) -- used as a second
# annotation track alongside region_layer on the heatmaps below. region_abb gives the short
# labels used for row/column names (matching Data/region_update.csv's Final_abb, the same
# convention used in EC_pseudotime_ridge.pdf).
cluster_lut <- obj_oi@meta.data %>% distinct(region_name, region_cluster)
stopifnot(!any(duplicated(cluster_lut$region_name)))
region_cluster_lookup <- setNames(as.character(cluster_lut$region_cluster), cluster_lut$region_name)
cluster_ord_layer  <- region_cluster_lookup[regions_ord]
cluster_cols_layer <- setNames(viridis(length(unique(cluster_ord_layer)), option = "turbo"), sort(unique(cluster_ord_layer)))
region_abb_vec <- setNames(region_meta$Final_abb[match(regions_ord, region_meta$Region_layer_1)], regions_ord)

mat_total <- matrix(NA_real_, nrow = K, ncol = K, dimnames = list(regions_pretty, regions_pretty))
mat_up    <- matrix(NA_real_, nrow = K, ncol = K, dimnames = list(regions_pretty, regions_pretty))
for (i in seq_len(nrow(summary_df))) {
  r1 <- summary_df$region1[i]; r2 <- summary_df$region2[i]
  mat_total[r1, r2] <- mat_total[r2, r1] <- summary_df$n_total[i]
  mat_up[r1, r2] <- summary_df$n_up_1v2[i]     # up in r1 vs r2
  mat_up[r2, r1] <- summary_df$n_down_1v2[i]   # up in r2 vs r1 (== down in r1 vs r2)
}
mat_down <- t(mat_up)   # down[row,col] = up[col,row], by construction

mat_total <- mat_total[regions_ord, regions_ord]
mat_up    <- mat_up[regions_ord, regions_ord]
mat_down  <- mat_down[regions_ord, regions_ord]

write.csv(mat_total, file.path(RES, "09_pairwise_DEG_matrix_total.csv"))
write.csv(mat_up,    file.path(RES, "09_pairwise_DEG_matrix_up.csv"))
write.csv(mat_down,  file.path(RES, "09_pairwise_DEG_matrix_down.csv"))


In [ ]:
Rscript_bin <- file.path(R.home("bin"), "Rscript")

# Renders a ComplexHeatmap to pdf+png in a fresh Rscript subprocess. Calling ComplexHeatmap::draw()
# directly inside this IRkernel session reliably corrupts/crashes on a repeat call (see
# 08_region_heterogeneity_dream_varpart.ipynb for the isolated repro) -- draw() is only ever called
# inside the subprocess here, and the resulting PNG issave_heatmap_files displayed via IRdisplay instead.
save_heatmap_files <- function(mat, row_anno = NULL, row_anno_cols = NULL, anno_name = "layer", legend_name, column_title,
                                pdf_path, png_path, col_fun_range = NULL, width = 15, height = 13.5) {
  bundle <- list(mat = mat, row_anno = row_anno, row_anno_cols = row_anno_cols, anno_name = anno_name,
                 legend_name = legend_name, column_title = column_title,
                 pdf_path = pdf_path, png_path = png_path, width = width, height = height,
                 col_fun_range = col_fun_range)
  bundle_path <- tempfile(fileext = ".rds")
  saveRDS(bundle, bundle_path)
  script_path <- tempfile(fileext = ".R")
  writeLines(c(
    'args <- commandArgs(trailingOnly=TRUE)',
    'b <- readRDS(args[1])',
    'suppressPackageStartupMessages({library(ComplexHeatmap); library(circlize)})',
    'rng <- b$col_fun_range',
    'if (is.null(rng)) rng <- c(0, max(b$mat, na.rm=TRUE))',
    'col_fun <- colorRamp2(c(rng[1], mean(rng), rng[2]), c("white", "#F4A582", "#B2182B"))',
    'ha <- NULL',
    'if (!is.null(b$row_anno)) {',
    '  al <- setNames(list(b$row_anno), b$anno_name); acl <- setNames(list(b$row_anno_cols), b$anno_name)',
    '  ha <- rowAnnotation(annotation_legend_param = list(), col = acl, annotation_name_gp = gpar(fontsize=8), df = setNames(data.frame(b$row_anno, stringsAsFactors=FALSE), b$anno_name))',
    '}',
    'ht <- Heatmap(b$mat, name = b$legend_name, col = col_fun, na_col = "grey92",',
    '              left_annotation = ha, cluster_rows = FALSE, cluster_columns = FALSE,',
    '              row_names_gp = gpar(fontsize = 7), column_names_gp = gpar(fontsize = 7),',
    '              column_title = b$column_title,',
    '              cell_fun = function(j, i, x, y, w, h, fill) {',
    '                v <- b$mat[i, j]',
    '                if (!is.na(v) && v > 0) grid.text(sprintf("%d", as.integer(v)), x, y, gp = gpar(fontsize = 4.5))',
    '              })',
    'pdf(b$pdf_path, width = b$width, height = b$height); draw(ht); dev.off()',
    'png(b$png_path, width = b$width, height = b$height, units = "in", res = 300); draw(ht); dev.off()'
  ), script_path)
  out <- system2(Rscript_bin, c(script_path, bundle_path), stdout = TRUE, stderr = TRUE)
  status <- attr(out, "status")
  if (!is.null(status) && status != 0) stop(paste(out, collapse = "\n"))
  invisible(NULL)
}

layer_cols <- setNames(viridis(length(unique(layer2_ord)), option = "turbo"), unique(layer2_ord))

# Dual-annotated variant: region_layer + region_cluster on both left and top, region_abb labels,
# larger cell-text font. Used for the region_layer-ordered heatmaps (order is NOT changed by
# adding the cluster track -- it's shown purely as a second annotation alongside region_layer).
save_dual_annot_heatmap <- function(mat, legend_name, column_title, pdf_path, png_path, width = 16, height = 14) {
  bundle <- list(mat = mat, layer_vec = layer2_ord, layer_cols = layer_cols,
                 cluster_vec = cluster_ord_layer, cluster_cols = cluster_cols_layer,
                 region_abb_vec = region_abb_vec, legend_name = legend_name, column_title = column_title,
                 pdf_path = pdf_path, png_path = png_path, width = width, height = height)
  bundle_path <- tempfile(fileext = ".rds")
  saveRDS(bundle, bundle_path)
  script_path <- tempfile(fileext = ".R")
  writeLines(c(
    'args <- commandArgs(trailingOnly=TRUE)',
    'b <- readRDS(args[1])',
    'suppressPackageStartupMessages({library(ComplexHeatmap); library(circlize)})',
    'rng <- c(0, max(b$mat, na.rm=TRUE))',
    'col_fun <- colorRamp2(c(rng[1], mean(rng), rng[2]), c("white", "#F4A582", "#B2182B"))',
    'ha_row <- rowAnnotation(region_layer = b$layer_vec, region_cluster = b$cluster_vec,',
    '                         col = list(region_layer = b$layer_cols, region_cluster = b$cluster_cols),',
    '                         annotation_name_gp = gpar(fontsize = 8))',
    'ha_top <- HeatmapAnnotation(region_layer = b$layer_vec, region_cluster = b$cluster_vec,',
    '                             col = list(region_layer = b$layer_cols, region_cluster = b$cluster_cols),',
    '                             annotation_name_gp = gpar(fontsize = 8), show_legend = c(FALSE, FALSE))',
    'ht <- Heatmap(b$mat, name = b$legend_name, col = col_fun, na_col = "grey92",',
    '              left_annotation = ha_row, top_annotation = ha_top,',
    '              cluster_rows = FALSE, cluster_columns = FALSE,',
    '              row_labels = b$region_abb_vec[rownames(b$mat)], column_labels = b$region_abb_vec[colnames(b$mat)],',
    '              row_names_gp = gpar(fontsize = 8), column_names_gp = gpar(fontsize = 8),',
    '              column_title = b$column_title,',
    '              cell_fun = function(j, i, x, y, w, h, fill) {',
    '                v <- b$mat[i, j]',
    '                if (!is.na(v) && v > 0) grid.text(sprintf("%d", as.integer(v)), x, y, gp = gpar(fontsize = 7))',
    '              })',
    'pdf(b$pdf_path, width = b$width, height = b$height); draw(ht); dev.off()',
    'png(b$png_path, width = b$width, height = b$height, units = "in", res = 300); draw(ht); dev.off()'
  ), script_path)
  out <- system2(Rscript_bin, c(script_path, bundle_path), stdout = TRUE, stderr = TRUE)
  status <- attr(out, "status")
  if (!is.null(status) && status != 0) stop(paste(out, collapse = "\n"))
  invisible(NULL)
}


In [ ]:
save_dual_annot_heatmap(mat_total, "Total DEGs", "Total DEGs per region pair (symmetric)",
                         file.path(FIG, "09_heatmap_total_DEGs.pdf"), file.path(FIG, "09_heatmap_total_DEGs.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_total_DEGs.png"))

In [ ]:
save_dual_annot_heatmap(mat_up, "Up DEGs", "Upregulated DEGs: row region vs column region",
                         file.path(FIG, "09_heatmap_up_DEGs.pdf"), file.path(FIG, "09_heatmap_up_DEGs.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_up_DEGs.png"))

In [ ]:
save_dual_annot_heatmap(mat_down, "Down DEGs", "Downregulated DEGs: row region vs column region",
                         file.path(FIG, "09_heatmap_down_DEGs.pdf"), file.path(FIG, "09_heatmap_down_DEGs.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_down_DEGs.png"))

### Same three heatmaps, regions grouped by `region_cluster` instead

`region_cluster` is the **data-driven** grouping used throughout this repo (`06.1`, `08`) — 4
clusters (`Cluster_1..4`) derived from capillary transcriptional/AV-zonation similarity, as
opposed to `Region_layer_2`'s gross anatomical grouping used above. It's a fixed per-region label
here (every `region_name` falls in exactly one `region_cluster`, confirmed below), so the same
`mat_total`/`mat_up`/`mat_down` matrices just need reordering — no new DEG tests.

In [ ]:
cluster_lut <- obj_oi@meta.data %>% distinct(region_name, region_cluster)
stopifnot(!any(duplicated(cluster_lut$region_name)))   # confirms region_cluster is fixed per region
region_cluster <- setNames(as.character(cluster_lut$region_cluster), cluster_lut$region_name)

ord_rc <- order(region_cluster[regions_pretty], regions_pretty)
regions_ord_rc <- regions_pretty[ord_rc]
cluster_ord   <- region_cluster[regions_ord_rc]
cat("region_cluster sizes:\n"); print(table(cluster_ord))

mat_total_rc <- mat_total[regions_ord_rc, regions_ord_rc]
mat_up_rc    <- mat_up[regions_ord_rc, regions_ord_rc]
mat_down_rc  <- mat_down[regions_ord_rc, regions_ord_rc]
cluster_cols <- setNames(viridis(length(unique(cluster_ord)), option = "turbo"), unique(cluster_ord))

In [ ]:
save_heatmap_files(mat_total_rc, row_anno = cluster_ord, row_anno_cols = cluster_cols, anno_name = "cluster",
                    legend_name = "Total DEGs", column_title = "Total DEGs per region pair, grouped by region_cluster",
                    pdf_path = file.path(FIG, "09_heatmap_total_DEGs_by_cluster.pdf"),
                    png_path = file.path(FIG, "09_heatmap_total_DEGs_by_cluster.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_total_DEGs_by_cluster.png"))

In [ ]:
save_heatmap_files(mat_up_rc, row_anno = cluster_ord, row_anno_cols = cluster_cols, anno_name = "cluster",
                    legend_name = "Up DEGs", column_title = "Upregulated DEGs (row vs column), grouped by region_cluster",
                    pdf_path = file.path(FIG, "09_heatmap_up_DEGs_by_cluster.pdf"),
                    png_path = file.path(FIG, "09_heatmap_up_DEGs_by_cluster.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_up_DEGs_by_cluster.png"))

In [ ]:
save_heatmap_files(mat_down_rc, row_anno = cluster_ord, row_anno_cols = cluster_cols, anno_name = "cluster",
                    legend_name = "Down DEGs", column_title = "Downregulated DEGs (row vs column), grouped by region_cluster",
                    pdf_path = file.path(FIG, "09_heatmap_down_DEGs_by_cluster.pdf"),
                    png_path = file.path(FIG, "09_heatmap_down_DEGs_by_cluster.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_down_DEGs_by_cluster.png"))

**Reading this against the anatomy-grouped version above**: `region_cluster` doesn't cleanly
separate high- from low-DEG pairs the way anatomical grouping loosely does — e.g. `Cluster_1`
(Pons, Spinal Cord, Midbrain, ...) still shows large within-cluster DEG counts (Posterior Cingulate
Cortex vs Spinal Cord: 112), and Superior Temporal Gyrus's own cluster-mates (`Cluster_3`:
Cingulum, Corpus Callosum, Lingual Gyrus, Periventricular White Matter, ...) don't shield it from
being the standout outlier — it still shows 100+ DEGs against Periventricular White Matter alone.
`region_cluster` was built to capture AV-zonation/capillary-composition similarity, not overall
transcriptional distance, so this isn't a contradiction, but it does mean the two groupings answer
different questions and shouldn't be read as interchangeable.

## 9.7 Most and least transcriptionally distinct region pairs

In [ ]:
cat("Top 15 most distinct pairs:\n")
print(summary_df %>% arrange(desc(n_total)) %>% head(15), row.names = FALSE)

cat("\nPairs with zero DEGs (", sum(summary_df$n_total == 0), " of ", nrow(summary_df), "):\n", sep="")
print(summary_df %>% filter(n_total == 0), row.names = FALSE)

**Superior Temporal Gyrus stands out**: it appears on one side of most of the top pairs above —
this single region drives an unusually large share of the strongest pairwise signal in this
dataset. The zero-DEG pairs, by contrast, cluster sensibly among anatomically/functionally related
regions (temporal-lobe pairs: Fusiform/Entorhinal/Inferior-Temporal/Parahippocampal Gyrus;
white-matter tracts: Corpus Callosum/Periventricular White Matter/White Matter Anterior Watershed;
frontal cortex: Orbitofrontal/Midfrontal/Superior-Frontal Gyrus; cingulate: Anterior/Posterior
Cingulate Cortex) — a sanity check that the pairwise signal tracks real regional similarity rather
than noise.

## 9.8 Functional summary A — per-region recurrent signature genes

Running GO enrichment separately on all 528 x 2 (up/down) gene sets isn't practical or
informative — most pairs have too few DEGs, and the output would be unreadable. Instead, for each
region a **recurrent signature** is built from the pairwise results themselves: a gene counts
toward region X's signature if it's significantly up in X in at least `WIN_FREQ_THRESH` of its 32
possible pairwise comparisons — i.e. genes that distinguish a region from many others, not just
one. Regions with at least `MIN_SIG_GENES_FOR_GO` such genes get a GO (Biological Process)
enrichment via `clusterProfiler::enrichGO`, exactly `06.1`'s own enrichment call.

In [ ]:
sig_df$up_region <- sub("^up_in_", "", sig_df$direction)
win_freq <- sig_df %>% count(gene, up_region, name = "n_wins")
signature_sets <- win_freq %>% filter(n_wins >= WIN_FREQ_THRESH) %>%
  group_by(up_region) %>% filter(n() >= MIN_SIG_GENES_FOR_GO) %>%
  summarise(genes = list(gene), n_genes = n(), .groups = "drop")
cat("regions with a qualifying signature (>=", MIN_SIG_GENES_FOR_GO, " genes at win-freq >=",
    WIN_FREQ_THRESH, "/32):", nrow(signature_sets), "of", K, "\n", sep="")
print(signature_sets %>% dplyr::select(up_region, n_genes) %>% arrange(desc(n_genes)), n = 40)

write.csv(signature_sets %>% dplyr::select(up_region, n_genes), file.path(RES, "09_region_signature_gene_counts.csv"), row.names = FALSE)
saveRDS(signature_sets, file.path(RES, "09_region_signature_gene_sets.rds"))

In [ ]:
t0 <- Sys.time()
ego_result <- data.frame()
for (r in signature_sets$up_region) {
  genes <- signature_sets$genes[[which(signature_sets$up_region == r)]]
  gene_ids <- mapIds(org.Hs.eg.db, keys = genes, keytype = "SYMBOL", column = "ENTREZID", multiVals = "first")
  gene_ids <- unique(gene_ids[!is.na(gene_ids)])
  if (length(gene_ids) < MIN_SIG_GENES_FOR_GO) next
  ego <- tryCatch(
    enrichGO(gene = gene_ids, OrgDb = org.Hs.eg.db, keyType = "ENTREZID", ont = "BP",
             pAdjustMethod = "BH", readable = TRUE, pvalueCutoff = 0.05, qvalueCutoff = 0.05),
    error = function(e) NULL)
  if (is.null(ego) || nrow(as.data.frame(ego)) == 0) next
  df <- as.data.frame(ego); df$region <- r
  ego_result <- rbind(ego_result, df)
}
cat("enrichGO total time:", round(as.numeric(Sys.time() - t0, units = "mins"), 1), "min\n")
cat("regions with >=1 significant GO BP term:", length(unique(ego_result$region)), "\n")
write.csv(ego_result, file.path(RES, "09_region_signature_GO_enrichment.csv"), row.names = FALSE)

In [ ]:
top_pathways <- ego_result %>%
  filter(p.adjust < 0.05) %>%
  group_by(region) %>%
  arrange(p.adjust) %>%
  slice_head(n = 3) %>%
  ungroup()

p_go <- ggplot(top_pathways, aes(x = region, y = reorder(Description, -p.adjust), fill = -log10(p.adjust))) +
  geom_tile() +
  scale_fill_viridis(option = "viridis") +
  labs(x = NULL, y = NULL, title = "Top 3 enriched GO Biological Process terms per region\n(recurrent pairwise-up signature genes)") +
  theme_classic(base_size = 9) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text.y = element_text(size = 7))
ggsave(file.path(FIG, "09_region_signature_GO_heatmap.pdf"), p_go, width = 11, height = 12)
ggsave(file.path(FIG, "09_region_signature_GO_heatmap.png"), p_go, width = 11, height = 12, dpi = 300)
p_go

## 9.9 Functional summary B — case studies: the most distinct region pairs

For the `N_CASE_STUDY_PAIRS` pairs with the most total DEGs, GO enrichment is run directly on that
pair's up-in-region1 and up-in-region2 gene sets — a pairwise-specific functional readout,
complementing the region-level signatures above.

In [ ]:
top_pairs <- summary_df %>% arrange(desc(n_total)) %>% head(N_CASE_STUDY_PAIRS)
print(top_pairs, row.names = FALSE)

t0 <- Sys.time()
pair_ego_result <- data.frame()
for (i in seq_len(nrow(top_pairs))) {
  r1 <- top_pairs$region1[i]; r2 <- top_pairs$region2[i]
  pair_label <- paste(r1, "vs", r2)
  for (target in c(r1, r2)) {
    genes <- sig_df %>% filter(region1 == r1, region2 == r2, direction == paste0("up_in_", target)) %>% pull(gene)
    if (length(genes) < MIN_SIG_GENES_FOR_GO) next
    gene_ids <- mapIds(org.Hs.eg.db, keys = genes, keytype = "SYMBOL", column = "ENTREZID", multiVals = "first")
    gene_ids <- unique(gene_ids[!is.na(gene_ids)])
    if (length(gene_ids) < MIN_SIG_GENES_FOR_GO) next
    ego <- tryCatch(
      enrichGO(gene = gene_ids, OrgDb = org.Hs.eg.db, keyType = "ENTREZID", ont = "BP",
               pAdjustMethod = "BH", readable = TRUE, pvalueCutoff = 0.05, qvalueCutoff = 0.05),
      error = function(e) NULL)
    if (is.null(ego) || nrow(as.data.frame(ego)) == 0) next
    df <- as.data.frame(ego); df$pair <- pair_label; df$up_in <- target
    pair_ego_result <- rbind(pair_ego_result, df)
  }
}
cat("pairwise case-study enrichGO time:", round(as.numeric(Sys.time() - t0, units = "mins"), 1), "min\n")
write.csv(pair_ego_result, file.path(RES, "09_case_study_pair_GO_enrichment.csv"), row.names = FALSE)

pair_ego_result %>%
  filter(p.adjust < 0.05) %>%
  group_by(pair, up_in) %>%
  arrange(p.adjust) %>%
  slice_head(n = 3) %>%
  ungroup() %>%
  dplyr::select(pair, up_in, Description, p.adjust, Count) %>%
  print(n = 40)

## 9.10 Overlap with curated BBB transport genes

Cross-referencing the pairwise significant DEGs against `Data/BBB_gene_list_updated_2026.csv`,
restricted to genes whose curated `primary_category` denotes an actual molecule-transport role
(`SLC transport`, `Active efflux`, `Receptor-mediated transport`, `Major facilitator`, `Channel`,
`Pump`, `Transcytosis`, `Receptor / transporter`) rather than structural/junction/signalling/marker
categories — i.e. genes that move specific substrates across the BBB, not barrier scaffolding or
zonation labels.

In [ ]:
TRANSPORT_CATEGORIES <- c("SLC transport", "Active efflux", "Receptor-mediated transport",
                           "Major facilitator", "Channel", "Pump", "Transcytosis",
                           "Transcytosis / enzyme", "Receptor / transporter")
glist <- read.csv(file.path(DATA, "BBB_gene_list_updated_2026.csv"))
transport_panel <- glist %>% filter(primary_category %in% TRANSPORT_CATEGORIES)
cat("curated transport genes (", length(TRANSPORT_CATEGORIES), " categories):", nrow(transport_panel), "of", nrow(glist), "\n", sep="")
print(table(transport_panel$primary_category))

hits <- sig_df %>% filter(gene %in% transport_panel$gene_symbol)
hits$up_region <- sub("^up_in_", "", hits$direction)
cat("\nsignificant gene-pair rows involving a transport gene:", nrow(hits), "of", nrow(sig_df), "\n")
cat("unique transport genes appearing as a pairwise DEG:", length(unique(hits$gene)), "of", nrow(transport_panel), "\n")

gene_rank <- hits %>% group_by(gene) %>%
  summarise(n_sig_pairs = n(), n_regions_up = n_distinct(up_region)) %>%
  arrange(desc(n_sig_pairs))
gene_rank <- merge(gene_rank, transport_panel[, c("gene_symbol","primary_category","subcategory","av_zonation","function_substrate")],
                    by.x = "gene", by.y = "gene_symbol")
gene_rank <- gene_rank[order(-gene_rank$n_sig_pairs), ]
write.csv(gene_rank, file.path(RES, "09_transport_marker_ranking.csv"), row.names = FALSE)
write.csv(hits, file.path(RES, "09_transport_marker_pairwise_hits.csv"), row.names = FALSE)
print(gene_rank[, c("gene","n_sig_pairs","n_regions_up","primary_category","subcategory","av_zonation")], row.names = FALSE)

**Which regions does each marker distinguish?** For the top markers, the regions they're
recurrently up in (with win count):

In [ ]:
top_markers <- head(gene_rank$gene, 10)
for (g in top_markers) {
  d <- hits %>% filter(gene == g) %>% count(up_region, sort = TRUE)
  cat(g, ": ", paste(paste0(d$up_region, " (", d$n, ")"), collapse = ", "), "\n", sep = "")
}

### Where each marker is distinctive, and how it's actually expressed

Two views of the same 28 genes: a **win-count** matrix (how many of a gene's significant pairwise
comparisons it wins in each region — a direct, significance-filtered specificity readout) and the
**balanced expression** pattern (donor+AV_bin-adjusted logCPM, z-scored — matching the residualization
convention used throughout this repo) across all 33 regions, for context.

In [ ]:
gene_rank
length(unique(gene_rank$gene))

In [ ]:
logcpm <- cpm(y, log = TRUE, prior.count = 1)
samp <- y$samples
donor_f <- factor(samp$donor); AV_bin_f <- factor(samp$AV_bin); region_f <- samp$region_name

balance_one <- function(g) {
  d <- data.frame(y = logcpm[g, ], donor = donor_f, AV_bin = AV_bin_f, region_name = region_f)
  m <- lm(y ~ donor + AV_bin, data = d)
  bal <- coef(m)["(Intercept)"] + resid(m)
  data.frame(gene = g, region_name = d$region_name, balanced = bal)
}
bal_long <- do.call(rbind, lapply(gene_rank$gene, balance_one))
bal_region <- bal_long %>% group_by(gene, region_name) %>% summarise(m = mean(balanced), .groups = "drop")
mat_marker <- tidyr::pivot_wider(bal_region, names_from = region_name, values_from = m) %>% as.data.frame()
rownames(mat_marker) <- mat_marker$gene; mat_marker$gene <- NULL
mat_marker <- as.matrix(mat_marker)[gene_rank$gene, , drop = FALSE]
matz_marker <- t(scale(t(mat_marker)))
write.csv(matz_marker, file.path(RES, "09_transport_marker_balanced_expression_zscore.csv"))

win_mat <- hits %>% count(gene, up_region) %>% tidyr::pivot_wider(names_from = up_region, values_from = n, values_fill = 0)
win_mat <- as.data.frame(win_mat); rownames(win_mat) <- win_mat$gene; win_mat$gene <- NULL
missing_regions <- setdiff(colnames(matz_marker), colnames(win_mat))
for (r in missing_regions) win_mat[[r]] <- 0
win_mat <- as.matrix(win_mat[gene_rank$gene, colnames(matz_marker)])
write.csv(win_mat, file.path(RES, "09_transport_marker_wincounts.csv"))
cat("win_mat:", nrow(win_mat), "genes x", ncol(win_mat), "regions | max wins:", max(win_mat), "\n")

In [ ]:
# region_layer (Region_layer_2) + region_abb (Final_abb), aligned to matz_marker/win_mat's column order
region_layer_vec <- setNames(region_meta$Region_layer_2[match(colnames(matz_marker), region_meta$Region_layer_1)], colnames(matz_marker))
region_abb_vec   <- setNames(region_meta$Final_abb[match(colnames(matz_marker), region_meta$Region_layer_1)], colnames(matz_marker))
gene_total_wins  <- setNames(gene_rank$n_sig_pairs, gene_rank$gene)[rownames(matz_marker)]   # per-gene win total (row sums of win_mat)
marker_layer_cols <- setNames(viridis(length(unique(region_layer_vec)), option = "turbo"), sort(unique(region_layer_vec)))

bundle_path <- tempfile(fileext = ".rds")
saveRDS(list(matz = matz_marker, win_mat = win_mat, region_layer_vec = region_layer_vec,
             region_abb_vec = region_abb_vec, gene_total_wins = gene_total_wins, layer_cols = marker_layer_cols),
        bundle_path)
script_path <- tempfile(fileext = ".R")
writeLines(c(
  'args <- commandArgs(trailingOnly=TRUE)',
  'b <- readRDS(args[1])',
  'suppressPackageStartupMessages({library(ComplexHeatmap); library(circlize)})',
  'col_fun_w <- colorRamp2(c(0, max(b$win_mat)), c("white", "#B2182B"))',
  'ht_w <- Heatmap(b$win_mat, name = "# wins", col = col_fun_w, cluster_rows = TRUE, cluster_columns = TRUE,',
  '                 row_names_gp = gpar(fontsize = 8), column_names_gp = gpar(fontsize = 8),',
  '                 column_labels = b$region_abb_vec[colnames(b$win_mat)],',
  '                 top_annotation = HeatmapAnnotation(region_layer = b$region_layer_vec, col = list(region_layer = b$layer_cols),',
  '                                                     annotation_name_gp = gpar(fontsize = 8)),',
  '                 column_title = "Pairwise wins per curated transport gene, by region",',
  '                 cell_fun = function(j, i, x, y, w, h, fill) {',
  '                   v <- b$win_mat[i, j]; if (v > 0) grid.text(sprintf("%d", v), x, y, gp = gpar(fontsize = 5))',
  '                 })',
  'pdf(file.path("../Figures", "09_transport_marker_wincounts_heatmap.pdf"), width = 13, height = 9.2); draw(ht_w); dev.off()',
  'png(file.path("../Figures", "09_transport_marker_wincounts_heatmap.png"), width = 13, height = 9.2, units = "in", res = 300); draw(ht_w); dev.off()',
  '',
  '# balanced-expression heatmap: + per-gene pairwise-win barplot (right), region_layer strip (top), region_abb labels',
  'col_fun_z <- colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))',
  'top_ha <- HeatmapAnnotation(region_layer = b$region_layer_vec, col = list(region_layer = b$layer_cols),',
  '                             annotation_name_gp = gpar(fontsize = 8),',
  '                             annotation_legend_param = list(region_layer = list(title = "region_layer")))',
  'right_ha <- rowAnnotation(`pairwise\nwins` = anno_barplot(b$gene_total_wins, gp = gpar(fill = "#B2182B", col = NA), width = unit(2.2, "cm")),',
  '                           annotation_name_gp = gpar(fontsize = 8, fontface = "italic"), annotation_name_rot = 90)',
  'ht_z <- Heatmap(b$matz, name = "z-score", col = col_fun_z,',
  '                 top_annotation = top_ha, right_annotation = right_ha,',
  '                 cluster_rows = TRUE, cluster_columns = TRUE,',
  '                 row_names_gp = gpar(fontsize = 8), column_names_gp = gpar(fontsize = 8),',
  '                 column_labels = b$region_abb_vec[colnames(b$matz)],',
  '                 column_title = "Balanced expression of curated transport genes, by region\nright: total pairwise wins per gene)")',
  'pdf(file.path("../Figures", "09_transport_marker_balanced_heatmap.pdf"), width = 13, height = 9.2); draw(ht_z); dev.off()',
  'png(file.path("../Figures", "09_transport_marker_balanced_heatmap.png"), width = 13, height = 9.2, units = "in", res = 300); draw(ht_z); dev.off()'
), script_path)
out <- system2(Rscript_bin, c(script_path, bundle_path), stdout = TRUE, stderr = TRUE)
status <- attr(out, "status")
if (!is.null(status) && status != 0) stop(paste(out, collapse = "\n"))
IRdisplay::display_png(file = file.path(FIG, "09_transport_marker_wincounts_heatmap.png"))

In [ ]:
IRdisplay::display_png(file = file.path(FIG, "09_transport_marker_balanced_heatmap.png"))

**Standout markers:**

- **`INSR`** (insulin receptor, a clinically active RMT drug-shuttle target) is up in **Pons in
  all 19 of its 19 significant pairs** — a perfectly region-specific marker in this dataset.
- **`SLC7A11`** (cystine/glutamate antiporter, ferroptosis resistance) splits cleanly between
  **Spinal Cord (25 wins)** and **Pons (19 wins)** — its 44 significant pairs are *entirely*
  accounted for by those two brainstem/spinal regions.
- **`BSG`** (basigin — MCT1/MCT4 chaperone and an RMT target) is dominated by **Precuneus (27 of
  45 wins)**.
- **`SLC44A1`** (choline transporter) and **`ALPL`** (alkaline phosphatase, the classic BBB
  histochemical marker) — the two top-recurrence markers overall — mark **Superior Temporal
  Gyrus** and **Pons** respectively, independently reproducing what `02_DE_regions_dream.ipynb`
  and `08_region_heterogeneity_dream_varpart.ipynb` already flagged as cross-donor-robust hits for
  these exact genes. Three independent notebooks in this repo now agree on both genes.
- Several of the clinically most important RMT "Trojan horse" targets for brain drug delivery —
  **`INSR`, `LDLR`, `LRP8`, `IGF1R`, `TFRC`** — all show real regional heterogeneity here. A
  delivery vehicle engineered against any one of them should be expected to reach different
  regions with different efficiency, not uniformly.
- In the balanced-expression view, the white-matter-tract regions (**Corpus Callosum, White
  Matter Anterior Watershed**, and to a lesser extent **Spinal Cord**) sit low across nearly the
  entire transport-gene panel — consistent with lower capillary transport specialization in white
  matter relative to grey-matter regions, though this dataset can't distinguish that from a
  vascular-density confound.

## 9.11 Sensitivity check — the same pairwise screen without AV_bin adjustment

A second, independent run of the whole pairwise analysis, dropping `AV_bin` from the model
entirely (`~0 + region_name + donor`, mirroring `06.1`'s own no-AV-control baseline section) —
does adjusting for residual AV zonation position change how many regional DEGs are found, and are
they the same genes? Pseudobulk is rebuilt at plain donor x region granularity (no AV_bin split,
same coverage filter as `08_region_heterogeneity_dream_varpart.ipynb`: >=3 donors with >=20 nuclei
per region) rather than reusing the AV_bin-split samples with AV_bin simply dropped from the
formula — splitting by AV_bin only to then not model it would just add avoidable pseudobulk
noise.

In [ ]:
tab_noav <- table(obj_oi$region_name, obj_oi$individualID)
passing_regions_noav <- sort(rownames(tab_noav)[rowSums(tab_noav >= 20) >= 3])
cat("regions passing coverage filter:", length(passing_regions_noav), "of", nrow(tab_noav), "\n")

scope_noav <- subset(obj_oi, subset = region_name %in% passing_regions_noav)
pseudo_noav <- AggregateExpression(scope_noav, assays = "RNA", return.seurat = TRUE,
                                    group.by = c("sampleID","region_name"))
mat_noav  <- pseudo_noav[['RNA']]$counts
meta_noav <- pseudo_noav@meta.data

donor_lut2 <- scope_noav@meta.data %>% distinct(sampleID, individualID)
donor_lut2$sampleID_dash <- gsub("_", "-", donor_lut2$sampleID)
meta_noav$donor <- donor_lut2$individualID[match(meta_noav$sampleID, donor_lut2$sampleID_dash)]
stopifnot(sum(is.na(meta_noav$donor)) == 0)

y_noav <- DGEList(counts = mat_noav, group = meta_noav$region_name)
y_noav$samples$donor <- meta_noav$donor
y_noav$samples$region_name <- meta_noav$region_name

cutoff_noav <- quantile(y_noav$samples$lib.size, LIBSIZE_QTILE_CUTOFF)
keep_noav <- y_noav$samples$lib.size > cutoff_noav
y_noav <- y_noav[, keep_noav]
cat("pseudobulk samples (no AV split, post lib.size filter):", ncol(y_noav), "\n")

group_noav <- droplevels(factor(y_noav$samples$region_name))
donor_noav <- droplevels(factor(y_noav$samples$donor))
cat("samples per region (min/median/max):", min(table(group_noav)), "/", median(table(group_noav)), "/", max(table(group_noav)), "\n")

# region_name -> pretty display name (make.names() mangles spaces to "." in design colnames,
# same fix as region_display in section 9.3 -- missed here in the first pass, which produced
# dotted region names in summary_df_noav that didn't match regions_pretty's spaced names).
region_display_noav <- levels(group_noav)
names(region_display_noav) <- make.names(region_display_noav)

design_noav <- model.matrix(~0 + group_noav + donor_noav)
colnames(design_noav) <- make.names(colnames(design_noav))
cat("design rank:", qr(design_noav)$rank, "of", ncol(design_noav), "\n")

keep_g <- filterByExpr(y_noav, design = design_noav)
cat("genes kept:", sum(keep_g), "of", length(keep_g), "(vs 2,001 in the AV_bin-adjusted model)\n")
y_noav <- y_noav[keep_g, , keep.lib.sizes = FALSE]
y_noav <- normLibSizes(y_noav)
y_noav <- estimateDisp(y_noav, design_noav, robust = TRUE)
fit_noav <- glmQLFit(y_noav, design_noav, robust = TRUE)
saveRDS(list(y = y_noav, fit = fit_noav, design = design_noav), file.path(RES, "09_fit_noAV.rds"))

In [ ]:
grp_cols_noav <- grep("^group_noav", colnames(design_noav), value = TRUE)
regions_noav  <- sub("^group_noav", "", grp_cols_noav)
pairs_noav <- combn(regions_noav, 2, simplify = FALSE)

mk_contrast_noav <- function(r1, r2, design) {
  v <- setNames(rep(0, ncol(design)), colnames(design))
  v[paste0("group_noav", r1)] <- 1
  v[paste0("group_noav", r2)] <- -1
  v
}

t0 <- Sys.time()
summary_rows_noav <- vector("list", length(pairs_noav))
sig_list_noav <- vector("list", length(pairs_noav))
for (k in seq_along(pairs_noav)) {
  r1 <- pairs_noav[[k]][1]; r2 <- pairs_noav[[k]][2]
  qlf <- glmQLFTest(fit_noav, contrast = mk_contrast_noav(r1, r2, design_noav))
  tt <- topTags(qlf, n = Inf, adjust.method = "BH")$table
  tt$gene <- rownames(tt)
  n_up   <- sum(tt$FDR < FDR_THRESH & tt$logFC >  LOGFC_THRESH)
  n_down <- sum(tt$FDR < FDR_THRESH & tt$logFC < -LOGFC_THRESH)
  summary_rows_noav[[k]] <- data.frame(region1 = region_display_noav[r1], region2 = region_display_noav[r2],
                                        n_up_1v2 = n_up, n_down_1v2 = n_down, n_total = n_up + n_down)
  sig <- tt[tt$FDR < FDR_THRESH & abs(tt$logFC) > LOGFC_THRESH, ]
  if (nrow(sig) > 0) {
    sig$region1 <- region_display_noav[r1]; sig$region2 <- region_display_noav[r2]
    sig$direction <- ifelse(sig$logFC > 0, paste0("up_in_", region_display_noav[r1]), paste0("up_in_", region_display_noav[r2]))
    sig_list_noav[[k]] <- sig
  }
}
cat("total time:", round(as.numeric(Sys.time() - t0, units = "mins"), 1), "min\n")

summary_df_noav <- do.call(rbind, summary_rows_noav)
sig_df_noav <- do.call(rbind, sig_list_noav)
cat("pairs with 0 DEGs (no AV adjustment):", sum(summary_df_noav$n_total == 0), "of", nrow(summary_df_noav), "\n")
cat("total significant gene-pair rows (no AV adjustment):", nrow(sig_df_noav), " (vs", nrow(sig_df), "AV-adjusted)\n")
print(summary(summary_df_noav$n_total))

write.csv(summary_df_noav, file.path(RES, "09_pairwise_DEG_summary_noAV.csv"), row.names = FALSE)
write.csv(sig_df_noav, file.path(RES, "09_pairwise_sig_genes_noAV.csv"), row.names = FALSE)

### Comparing the two models

Two things to check: does the **total DEG count per pair** change with AV adjustment, and — for
pairs/genes tested under **both** models — do the **effect sizes (logFC) agree**? Genes here are
matched by `(gene, region1, region2)` since the two models were fit on different pseudobulk/gene
sets.

In [ ]:
cmp <- merge(summary_df[, c("region1","region2","n_total")], summary_df_noav[, c("region1","region2","n_total")],
             by = c("region1","region2"), suffixes = c("_AV",""))
colnames(cmp)[colnames(cmp) == "n_total"] <- "n_total_noAV"
cat("total DEGs summed across all pairs -- AV-adjusted:", sum(cmp$n_total_AV), " | no-AV:", sum(cmp$n_total_noAV), "\n")
cat("pearson correlation of per-pair total DEG counts:", round(cor(cmp$n_total_AV, cmp$n_total_noAV), 3), "\n")
cat("median per-pair change (no-AV minus AV-adjusted):", median(cmp$n_total_noAV - cmp$n_total_AV), "\n")

## logFC agreement for the same (gene, pair) tested significant under either model
sig_df_noav$region1_r <- pmin(sig_df_noav$region1, sig_df_noav$region2)
sig_df_noav$region2_r <- pmax(sig_df_noav$region1, sig_df_noav$region2)
sig_df_r <- sig_df
sig_df_r$region1_r <- pmin(sig_df_r$region1, sig_df_r$region2)
sig_df_r$region2_r <- pmax(sig_df_r$region1, sig_df_r$region2)
both <- merge(sig_df_r[, c("gene","region1_r","region2_r","logFC")],
              sig_df_noav[, c("gene","region1_r","region2_r","logFC")],
              by = c("gene","region1_r","region2_r"), suffixes = c("_AV","_noAV"))
cat("\ngene-pair hits significant under BOTH models:", nrow(both), "\n")
cat("logFC correlation (AV-adjusted vs no-AV, shared significant hits):", round(cor(both$logFC_AV, both$logFC_noAV), 3), "\n")

## are the 28 transport markers still significant without AV adjustment?
transport_noav <- sig_df_noav %>% filter(gene %in% gene_rank$gene) %>% count(gene, name = "n_sig_pairs_noAV")
transport_cmp <- merge(gene_rank[, c("gene","n_sig_pairs")], transport_noav, by = "gene", all.x = TRUE)
transport_cmp$n_sig_pairs_noAV[is.na(transport_cmp$n_sig_pairs_noAV)] <- 0
transport_cmp <- transport_cmp[order(-transport_cmp$n_sig_pairs), ]
colnames(transport_cmp)[colnames(transport_cmp) == "n_sig_pairs"] <- "n_sig_pairs_AVadjusted"
write.csv(transport_cmp, file.path(RES, "09_transport_marker_AV_sensitivity.csv"), row.names = FALSE)
print(transport_cmp, row.names = FALSE)

In [ ]:
mat_total_noav <- matrix(NA_real_, nrow = length(regions_pretty), ncol = length(regions_pretty),
                          dimnames = list(regions_pretty, regions_pretty))
for (i in seq_len(nrow(summary_df_noav))) {
  r1 <- summary_df_noav$region1[i]; r2 <- summary_df_noav$region2[i]
  mat_total_noav[r1, r2] <- mat_total_noav[r2, r1] <- summary_df_noav$n_total[i]
}
mat_total_noav <- mat_total_noav[regions_ord, regions_ord]   # same anatomy ordering as section 9.6

save_dual_annot_heatmap(mat_total_noav, "Total DEGs", "Total DEGs per region pair, WITHOUT AV_bin adjustment",
                         file.path(FIG, "09_heatmap_total_DEGs_noAV.pdf"), file.path(FIG, "09_heatmap_total_DEGs_noAV.png"))
IRdisplay::display_png(file = file.path(FIG, "09_heatmap_total_DEGs_noAV.png"))

Same anatomical ordering as the AV-adjusted heatmap in §9.6, for direct visual comparison —
Superior Temporal Gyrus's dark band should look familiar if AV adjustment isn't the main driver of
that pattern.

## Summary

In [ ]:
cat("Step 9 -- pairwise DEG analysis, 33 regions x", length(pairs), "pairs, edgeR (glmQLFit/glmQLFTest), donor+AV_bin adjusted\n")
cat("Pseudobulk samples:", ncol(y), " | genes tested:", nrow(y), "\n")
cat("Pairs with >=1 DEG:", sum(summary_df$n_total>0), "of", nrow(summary_df),
    " | median DEGs/pair:", median(summary_df$n_total), " | max:", max(summary_df$n_total), "\n")
cat("Total significant gene-pair rows:", nrow(sig_df), "\n")
cat("Regions with a qualifying recurrent signature:", nrow(signature_sets), "of", K, "\n")
cat("Curated transport genes appearing as pairwise DEGs:", length(unique(hits$gene)), "of", nrow(transport_panel), "\n")
cat("Sensitivity check (no AV_bin adjustment): total DEG-pair sum AV-adjusted =", sum(cmp$n_total_AV),
    " vs no-AV =", sum(cmp$n_total_noAV), "; per-pair count correlation =", round(cor(cmp$n_total_AV, cmp$n_total_noAV), 3), "\n")